# **Child Well-being - Data Transformer**<br/>
**University: University of Milano-Bicocca**<br/>
**Master's Degree: Data Science (A.Y. 2025/2026)**<br/>
**Course: Data Science Lab**<br/>

In [3]:
import sys
sys.path.insert(0, "..")

import polars as pl
import numpy as np

from scripts.transformer import normalize_minmax, discretize

Structure of the INDICATORS dictionary:

 - description: short description of the indicator
 - dimension: macro-dimension (A=Outcomes, B=Relationships and Social Context, C=Policies and Resources)
 - subdimension: sub-dimension associated to the indicator
 - measure_type: type of measure (e.g., percentage, rate, expenditure per capita, etc.)
 - direction: "positive" if high values = more well-being, "negative" if high values = less well-being
 - age_group: age group of reference
 - source: main source of the data

 Min-max normalization is applied to all indicators, with the following formulas:
 - Positive direction (direct):   (x - min) / (max - min)
 - Negative direction (inverted): (max - x) / (max - min)

Result: all variables normalized in [0, 1] with 1 = maximum well-being.

In [ ]:
INDICATORS: dict[str, dict] = {
    # ===== A — Outcomes =====
    # A1 — Material conditions
    "A1_2": {
        "description": "Severe housing deprivation (0-17 years)",
        "dimension": "A",
        "subdimension": "A1_material_conditions",
        "measure_type": "percentage",
        "direction": "negative",
        "age_group": "0-17",
        "source": "Eurostat / EU-SILC",
    },
    # A2 — Physical health conditions
    "A2_1": {
        "description": "Infant mortality (under 1 year)",
        "dimension": "A",
        "subdimension": "A2_health",
        "measure_type": "rate",
        "direction": "negative",
        "age_group": "<1",
        "source": "OECD Health Statistics",
    },
    # A3 — Cognitive & educational outcome
    "A3_3": {
        "description": "Top performers PISA (level 5-6 in at least one subject)",
        "dimension": "A",
        "subdimension": "A3_education",
        "measure_type": "percentage",
        "direction": "positive",
        "age_group": "15",
        "source": "PISA",
    },
    "A3_4": {
        "description": "Students who expect to complete tertiary education",
        "dimension": "A",
        "subdimension": "A3_education",
        "measure_type": "percentage",
        "direction": "positive",
        "age_group": "15",
        "source": "PISA",
    },
    "A3_5": {
        "description": "NEET (15-29 years)",
        "dimension": "A",
        "subdimension": "A3_education",
        "measure_type": "percentage",
        "direction": "negative",
        "age_group": "15-29",
        "source": "OECD / Eurostat",
    },
    # A4 — Social & emotional outcome
    "A4_6": {
        "description": "High life satisfaction (score 9-10)",
        "dimension": "A",
        "subdimension": "A4_subjective_wellbeing",
        "measure_type": "percentage",
        "direction": "positive",
        "age_group": "15",
        "source": "PISA",
    },

    # ===== B — Relationships and Social Context =====
    # B1 — Home & Family Life
    "B1_1": {
        "description": "Relative child poverty (50% median threshold)",
        "dimension": "B",
        "subdimension": "B1_family_income",
        "measure_type": "percentage",
        "direction": "negative",
        "age_group": "0-17",
        "source": "OECD / EU-SILC",
    },
    "B1_5": {
        "description": "Parents encourage self-confidence (strongly agree)",
        "dimension": "B",
        "subdimension": "B1_family_income",
        "measure_type": "percentage",
        "direction": "positive",
        "age_group": "15",
        "source": "PISA",
    },
    # B2 — Life at school & early education and care
    "B2_4": {
        "description": "Bullying at school (at least once a month)",
        "dimension": "B",
        "subdimension": "B2_school_early_childhood",
        "measure_type": "percentage",
        "direction": "negative",
        "age_group": "15",
        "source": "PISA",
    },
    "B2_5": {
        "description": "Sense of belonging to school (agree/strongly agree)",
        "dimension": "B",
        "subdimension": "B2_school_early_childhood",
        "measure_type": "percentage",
        "direction": "positive",
        "age_group": "15",
        "source": "PISA",
    },
    # B3 — Social Life and life in the community
    "B3_5": {
        "description": "Criminality/violence in the area of residence",
        "dimension": "B",
        "subdimension": "B3_safety",
        "measure_type": "percentage",
        "direction": "negative",
        "age_group": "0-17",
        "source": "EU-SILC",
    },
    # B4 — Online life
    "B4_3": {
        "description": "Internet as a major source of information (strongly agree)",
        "dimension": "B",
        "subdimension": "B4_digital",
        "measure_type": "percentage",
        "direction": "positive",
        "age_group": "15",
        "source": "PISA",
    },

    # ===== C — Policies and Resources =====
    "C1_2": {
        "description": "Reduction in child poverty from taxes/transfers (pp)",
        "dimension": "C",
        "subdimension": "C1_family_spending",
        "measure_type": "percentage_points",
        "direction": "positive",
        "age_group": "0-17",
        "source": "OECD",
    },
    "C1_3": {
        "description": "Minimum guaranteed income for couples with 2 children (% median)",
        "dimension": "C",
        "subdimension": "C1_family_spending",
        "measure_type": "percentage",
        "direction": "positive",
        "age_group": "n/a",
        "source": "OECD Tax-Benefit Model",
    },
    "C1_4": {
        "description": "Maternity leave with pay (total weeks)",
        "dimension": "C",
        "subdimension": "C1_family_spending",
        "measure_type": "weeks",
        "direction": "positive",
        "age_group": "n/a",
        "source": "OECD Family Database",
    },
    "C1_5": {
        "description": "Paternity leave with pay (total weeks)",
        "dimension": "C",
        "subdimension": "C1_family_spending",
        "measure_type": "weeks",
        "direction": "positive",
        "age_group": "n/a",
        "source": "OECD Family Database",
    },
    # C2 — House & Community policies
    "C2_1": {
        "description": "Public expenditure on housing and community services per person",
        "dimension": "C",
        "subdimension": "C2_housing_culture",
        "measure_type": "currency_per_capita",
        "direction": "positive",
        "age_group": "n/a",
        "source": "OECD / COFOG",
    },
    "C2_2": {
        "description": "Public expenditure on recreation, culture and religion per person",
        "dimension": "C",
        "subdimension": "C2_housing_culture",
        "measure_type": "currency_per_capita",
        "direction": "positive",
        "age_group": "n/a",
        "source": "OECD / COFOG",
    },
    "C2_3": {
        "description": "Public expenditure on social protection and housing per person",
        "dimension": "C",
        "subdimension": "C2_housing_culture",
        "measure_type": "currency_per_capita",
        "direction": "positive",
        "age_group": "n/a",
        "source": "OECD / COFOG",
    },
    # C3 — Health policies
    "C3_1": {
        "description": "Public expenditure on public health per person",
        "dimension": "C",
        "subdimension": "C3_public_health",
        "measure_type": "currency_per_capita",
        "direction": "positive",
        "age_group": "n/a",
        "source": "OECD Health Statistics",
    },
    "C3_2": {
        "description": "Vaccination DPT 3 doses (% under 1 year)",
        "dimension": "C",
        "subdimension": "C3_public_health",
        "measure_type": "percentage",
        "direction": "positive",
        "age_group": "<1",
        "source": "OECD Health Statistics",
    },
    "C3_3": {
        "description": "Vaccination measles at least 1 dose (% under 1 year)",
        "dimension": "C",
        "subdimension": "C3_public_health",
        "measure_type": "percentage",
        "direction": "positive",
        "age_group": "<1",
        "source": "OECD Health Statistics",
    },
    # C4 — Education & early childhood policies
    "C4_2": {
        "description": "Net cost of childcare per parent (low earning couple, 2 children)",
        "dimension": "C",
        "subdimension": "C4_education_childcare",
        "measure_type": "percentage",
        "direction": "negative",
        "age_group": "n/a",
        "source": "OECD Tax-Benefit Model",
    },
    "C4_4": {
        "description": "Public expenditure on primary and secondary education per student FTE",
        "dimension": "C",
        "subdimension": "C4_education_childcare",
        "measure_type": "currency_per_capita",
        "direction": "positive",
        "age_group": "primary-secondary",
        "source": "OECD Education at a Glance",
    },
    "C4_5": {
        "description": "Public expenditure on ancillary education services per student FTE",
        "dimension": "C",
        "subdimension": "C4_education_childcare",
        "measure_type": "currency_per_capita",
        "direction": "positive",
        "age_group": "primary-secondary",
        "source": "OECD Education at a Glance",
    },
    "C4_6": {
        "description": "Student-teacher ratio secondary (FTE)",
        "dimension": "C",
        "subdimension": "C4_education_childcare",
        "measure_type": "ratio",
        "direction": "negative",
        "age_group": "secondary",
        "source": "OECD Education at a Glance",
    },
    # C5 — Environmental policies
    "C5_1": {
        "description": "Public expenditure on environment per person",
        "dimension": "C",
        "subdimension": "C5_environment",
        "measure_type": "currency_per_capita",
        "direction": "positive",
        "age_group": "n/a",
        "source": "OECD / COFOG",
    },
}

ALL_INDICATORS = list(INDICATORS.keys())

NEGATIVE_INDICATORS = [k for k, v in INDICATORS.items() if v["direction"] == "negative"]
POSITIVE_INDICATORS = [k for k, v in INDICATORS.items() if v["direction"] == "positive"]

DIMENSIONS = sorted(set(v["dimension"] for v in INDICATORS.values()))
SUBDIMENSIONS = sorted(set(v["subdimension"] for v in INDICATORS.values()))

INDICATORS_BY_DIMENSION = {
    dim: [k for k, v in INDICATORS.items() if v["dimension"] == dim]
    for dim in DIMENSIONS
}

INDICATORS_BY_SUBDIMENSION = {
    sub: [k for k, v in INDICATORS.items() if v["subdimension"] == sub]
    for sub in SUBDIMENSIONS
}

print(f"Total indicators: {len(ALL_INDICATORS)}")
print(f"  - positive direction: {len(POSITIVE_INDICATORS)}")
print(f"  - negative direction: {len(NEGATIVE_INDICATORS)}")
print(f"Dimensions: {DIMENSIONS}")
print(f"Sub-dimensions: {SUBDIMENSIONS}")
print()

for dim, cols in INDICATORS_BY_DIMENSION.items():
    print(f"[{dim}] ({len(cols)} indicators): {cols}")

print("\nIndicators to invert:")
for ind in NEGATIVE_INDICATORS:
    print(f"  {ind}: {INDICATORS[ind]['description']}")

print("\nIndicators to keep:")
for ind in POSITIVE_INDICATORS:
    print(f"  {ind}: {INDICATORS[ind]['description']}")

Total indicators: 27
  - positive direction: 19
  - negative direction: 8
Dimensions: ['A', 'B', 'C']
Sub-dimensions: ['A1_material_conditions', 'A2_health', 'A3_education', 'A4_subjective_wellbeing', 'B1_family_income', 'B2_school_early_childhood', 'B3_safety', 'B4_digital', 'C1_family_spending', 'C2_housing_culture', 'C3_public_health', 'C4_education_childcare', 'C5_environment']

[A] (6 indicators): ['A1_2', 'A2_1', 'A3_3', 'A3_4', 'A3_5', 'A4_6']
[B] (6 indicators): ['B1_1', 'B1_5', 'B2_4', 'B2_5', 'B3_5', 'B4_3']
[C] (15 indicators): ['C1_2', 'C1_3', 'C1_4', 'C1_5', 'C2_1', 'C2_2', 'C2_3', 'C3_1', 'C3_2', 'C3_3', 'C4_2', 'C4_4', 'C4_5', 'C4_6', 'C5_1']

Indicators to invert:
  A1_2: Severe housing deprivation (0-17 years)
  A2_1: Infant mortality (under 1 year)
  A3_5: NEET (15-29 years)
  B1_1: Relative child poverty (50% median threshold)
  B2_4: Bullying at school (at least once a month)
  B3_5: Criminality/violence in the area of residence
  C4_2: Net cost of childcare per par

In [5]:
df = pl.read_parquet('../data/020_child_well_being.parquet')

In [6]:
df = normalize_minmax(
    df,
    indicators=INDICATORS,
    suffix="_norm",
)
df = discretize(
    df,
    columns=ALL_INDICATORS,
    n_levels=4,
    normalized_suffix="_norm",
)

In [7]:
# Drop the original non-normalized columns and the normalized columns with the suffix "_norm"
# Rename the discretized columns to remove the suffix "_ord"

df_out = df.drop(
    [col for col in df.columns if col.endswith("_norm") or col in ALL_INDICATORS]
).rename(
    {col: col.replace("_ord", "") for col in df.columns if col.endswith("_ord")}
)

In [8]:
df_out.write_parquet('../data/030_child_well_being.parquet')

### Analysis to apply other filters

In [88]:
df_out

REF_AREA,TIME_PERIOD,A1_2,A2_1,A3_3,A3_4,A3_5,A4_6,B1_1,B1_5,B2_4,B2_5,B3_5,B4_3,C1_2,C1_3,C1_4,C1_5,C2_1,C2_2,C2_3,C3_1,C3_2,C3_3,C4_2,C4_4,C4_5,C4_6,C5_1
str,i64,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32
"""SWE""",2018,2,1,4,4,1,2,1,2,2,2,4,2,2,2,2,4,4,4,4,4,3,3,1,4,4,3,2
"""SWE""",2015,1,1,3,2,1,null,1,4,1,2,3,3,2,2,3,4,4,4,4,4,4,3,1,4,4,4,2
"""LTU""",2018,4,3,1,4,1,4,4,3,3,1,1,2,3,4,3,3,1,2,2,1,1,1,3,1,1,1,1
"""LTU""",2015,4,4,1,4,3,4,4,4,1,1,1,3,3,3,3,3,1,1,2,1,1,2,3,1,1,1,1
"""LVA""",2018,4,2,1,3,2,3,2,1,4,3,2,1,2,2,3,1,4,2,2,1,2,3,3,1,1,1,1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""CZE""",2015,2,1,2,3,2,1,2,1,3,3,4,2,2,1,4,1,3,2,4,2,3,4,4,1,2,3,4
"""SVK""",2018,2,4,2,3,3,4,3,1,4,2,1,1,1,1,4,1,2,2,1,2,3,2,2,2,3,4,2
"""SVK""",2015,3,4,1,null,4,4,3,1,3,2,1,1,1,1,4,1,3,1,1,2,2,2,2,1,3,3,3


In [89]:
len(df_out.columns)

29

In [90]:
df_out.describe()

statistic,REF_AREA,TIME_PERIOD,A1_2,A2_1,A3_3,A3_4,A3_5,A4_6,B1_1,B1_5,B2_4,B2_5,B3_5,B4_3,C1_2,C1_3,C1_4,C1_5,C2_1,C2_2,C2_3,C3_1,C3_2,C3_3,C4_2,C4_4,C4_5,C4_6,C5_1
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""","""32""",32.0,32.0,32.0,31.0,31.0,32.0,31.0,32.0,32.0,31.0,32.0,32.0,31.0,32.0,32.0,32.0,32.0,32.0,32.0,32.0,32.0,32.0,32.0,32.0,32.0,32.0,31.0,32.0
"""null_count""","""0""",0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
"""mean""",null,2016.5,2.4375,2.34375,2.419355,2.419355,2.4375,2.419355,2.4375,2.4375,2.387097,2.40625,2.4375,2.387097,2.4375,2.375,2.40625,2.375,2.4375,2.4375,2.4375,2.4375,2.40625,2.1875,2.28125,2.4375,2.4375,2.419355,2.4375
"""std""",null,1.524002,1.162242,1.180743,1.148164,1.148164,1.162242,1.148164,1.162242,1.162242,1.115868,1.131923,1.162242,1.115868,1.162242,1.128802,1.131923,1.184578,1.162242,1.162242,1.162242,1.162242,1.160072,1.119836,1.224333,1.162242,1.162242,1.148164,1.162242
"""min""","""AUT""",2015.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
"""25%""",null,2015.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
"""50%""",null,2018.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0
"""75%""",null,2018.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0
"""max""","""SWE""",2018.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0


In [91]:
countries = df_out.select(pl.col("REF_AREA")).to_series().to_list()
indicators_list = list(INDICATORS.keys())

In [ ]:
null_counts = {}
for country in countries:
    #print(f"Country: {country}")
    df_country = df_out.filter(pl.col("REF_AREA") == country)
    for metric in indicators_list:
        # count the number of null values for the metric in the country
        null_count = df_country.select(pl.col(metric)).null_count().to_series()[0]
        #print(f"  {metric}: {null_count} null values")
        if null_count > 0 and metric not in null_counts:
            null_counts[(metric)] = [country]
        elif null_count > 0 and metric in null_counts:
            null_counts[(metric)].append(country)

In [93]:
null_counts

{'A4_6': ['SWE', 'SWE'],
 'A3_3': ['ESP', 'ESP'],
 'B4_3': ['PRT', 'PRT'],
 'B2_4': ['ITA', 'ITA'],
 'C4_6': ['SVN', 'SVN'],
 'A3_4': ['SVK', 'SVK']}

In [ ]:
null_countries = dict()
for key, value in null_counts.items():
    for country in value:
        if country not in null_countries:
            null_countries[country] = 1
        null_countries[country] += 1

null_indicators = dict()
for key, value in null_counts.items():
    null_indicators[key] = len(value)

In [95]:
null_countries

{'SWE': 3, 'ESP': 3, 'PRT': 3, 'ITA': 3, 'SVN': 3, 'SVK': 3}

In [96]:
null_indicators

{'A4_6': 2, 'A3_3': 2, 'B4_3': 2, 'B2_4': 2, 'C4_6': 2, 'A3_4': 2}

*This notebook is licensed under [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/).*